In [2]:
pip install upstox-python-sdk xlwings pandas python-dotenv
# pip install appscript psutil

SyntaxError: invalid syntax (1168657649.py, line 1)

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)  # settings live in .env (see .env.example)

import upstox_client

# Credentials are read from .env (UPSTOX_API_KEY / UPSTOX_REDIRECT_URI)
api_key = os.getenv("UPSTOX_API_KEY")
redirect_uri = os.getenv("UPSTOX_REDIRECT_URI", "https://127.0.0.1:5000/")  # must match the Upstox app settings

# Step 1: Create the Login URL
print(f"Go to this URL in your browser:\n")
print(f"https://api.upstox.com/v2/login/authorization/dialog?client_id={api_key}&redirect_uri={redirect_uri}&response_type=code")

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)  # settings live in .env (see .env.example)

import requests
from dotenv import set_key

# The code changes on every login: paste the code=... part of the redirected URL
auth_code = input("Paste the code from the redirect URL: ").strip()
api_secret = os.getenv("UPSTOX_API_SECRET")

url = 'https://api.upstox.com/v2/login/authorization/token'
headers = {
    'accept': 'application/json',
    'Content-Type': 'application/x-www-form-urlencoded',
}

data = {
    'code': auth_code,
    'client_id': api_key,
    'client_secret': api_secret,
    'redirect_uri': redirect_uri,
    'grant_type': 'authorization_code',
}

response = requests.post(url, headers=headers, data=data)
ACCESS_TOKEN = response.json()['access_token']
set_key(".env", "UPSTOX_ACCESS_TOKEN", ACCESS_TOKEN)  # reused by the other notebooks + Streamlit app
print("Success! Access token saved to .env")


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [8]:
pip install appscript psutil upstox-python-sdk xlwings pandas python-dotenv


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)  # settings live in .env (see .env.example)

import xlwings as xw
import pandas as pd
import time
from datetime import datetime
import sys

# Ensure dependencies are loaded
try:
    import appscript
    import psutil
except ImportError:
    print("Dependencies missing. Please run: pip install appscript psutil")

from upstox_client import ApiClient, Configuration, MarketDataStreamerV3

# --- CONFIGURATION ---
ACCESS_TOKEN = os.getenv("UPSTOX_ACCESS_TOKEN") or globals().get("ACCESS_TOKEN")  # saved by the login cell / Streamlit app
NIFTY_50_URL = "https://archives.nseindia.com/content/indices/ind_nifty50list.csv"

# 1. MAC-SPECIFIC EXCEL LAUNCHER
def connect_to_excel():
    print("Connecting to Excel...")
    try:
        # Check if Excel is even running
        is_excel_running = "Microsoft Excel" in [p.info['name'] for p in psutil.process_iter(['name'])]
        
        if not is_excel_running:
            print("Launching Microsoft Excel...")
            app = xw.App(visible=True)
            wb = app.books.add()
        else:
            # Connect to existing app
            wb = xw.Book()
        
        return wb
    except Exception as e:
        print(f"Excel Connection Error: {e}")
        print("Manual Fix: Open a blank Excel file manually, then re-run this script.")
        return None

wb = connect_to_excel()
if wb:
    sheet = wb.sheets[0]
    sheet.clear()
    headers = ["Symbol", "LTP", "5m High", "5m Low", "Candle Start", "Last Update"]
    sheet.range('A1').value = headers
    sheet.range('A1:F1').api.font_object.bold.set(True) # Make headers bold

# 2. DATA HANDLING
def get_nifty_keys():
    try:
        df = pd.read_csv(NIFTY_50_URL)
        return [f"NSE_EQ|{s}" for s in df['Symbol'].tolist()]
    except:
        # Fallback if NSE website is down
        return ["NSE_EQ|RELIANCE", "NSE_EQ|TCS", "NSE_EQ|INFY"] 

instrument_keys = get_nifty_keys()
data_store = {k: {'high': 0, 'low': 999999, 'ltp': 0, 'start_time': ""} for k in instrument_keys}

# 3. WEBSOCKET LOGIC
def on_market_update(data):
    now = datetime.now()
    # Logic for 5-minute candle synchronization
    bucket_min = (now.minute // 5) * 5
    candle_label = now.replace(minute=bucket_min, second=0, microsecond=0).strftime('%H:%M')

    for key, values in data['feeds'].items():
        if 'ltp' not in values: continue
        ltp = values['ltp']
        
        # New candle reset
        if data_store[key]['start_time'] != candle_label:
            data_store[key]['high'] = ltp
            data_store[key]['low'] = ltp
            data_store[key]['start_time'] = candle_label
        else:
            if ltp > data_store[key]['high']: data_store[key]['high'] = ltp
            if ltp < data_store[key]['low']: data_store[key]['low'] = ltp
        
        data_store[key]['ltp'] = ltp

# 4. START THE ENGINE
conf = Configuration()
conf.access_token = ACCESS_TOKEN
streamer = MarketDataStreamerV3(ApiClient(conf))

streamer.on_open = lambda: streamer.subscribe(instrument_keys, mode="full")
streamer.on_message = on_market_update

print("Starting Stream...")
streamer.connect()

try:
    while True:
        # Update Excel every 2 seconds
        if any(v['ltp'] > 0 for v in data_store.values()):
            output = []
            for key in instrument_keys:
                d = data_store[key]
                output.append([key.split('|')[1], d['ltp'], d['high'], d['low'], d['start_time'], datetime.now().strftime('%H:%M:%S')])
            sheet.range('A2').value = output
        time.sleep(2)
except KeyboardInterrupt:
    streamer.disconnect()
    print("Stream Disconnected.")

In [10]:
import sys
!{sys.executable} -m pip install appscript psutil


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
